# Train the 5-object detector (can, glasses, marker, watch, water bottle)

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

Step 2 asks you to pick `combined_5_objects_dataset.zip` from
`Object_Detection\datasets\` on your PC. Training takes about 1 hour on a T4.
At the end the browser downloads **`5_objects_detector.onnx`**. Put it in
`Object_Detection\models\` (back up the old one first).

## 1. Check the GPU and install Ultralytics

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "ultralytics>=8.3,<9"
import ultralytics; ultralytics.checks()

## 2. Upload the dataset zip
If the upload is slow, put the zip in Google Drive instead and set `DRIVE_ZIP` to its path.

In [ ]:
import os, zipfile, shutil
DRIVE_ZIP = ""   # e.g. "/content/drive/MyDrive/combined_5_objects_dataset.zip"

if DRIVE_ZIP:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = DRIVE_ZIP
else:
    from google.colab import files
    uploaded = files.upload()
    zip_path = next(iter(uploaded))

shutil.rmtree("/content/dataset", ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/dataset")
print(open("/content/dataset/data.yaml").read())
for s in ("train", "valid"):
    print(s, len(os.listdir(f"/content/dataset/{s}/images")), "images")

## 3. Train
`yolo11s` is more accurate than the nano model the old detector used, and still runs well on
an RTX 4050. `patience=30` stops early when the model stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
results = model.train(
    data="/content/dataset/data.yaml",
    imgsz=640, epochs=150, patience=30, batch=16,
    degrees=10, scale=0.5, hsv_v=0.5, close_mosaic=15,
    name="five_objects", exist_ok=True,
)
BEST = "/content/runs/detect/five_objects/weights/best.pt"

## 4. Results
The first number is on the whole validation set. The second uses **only your own webcam
photos**, so it is the one that tells you how it will do live.

In [ ]:
import glob
best = YOLO(BEST)
m = best.val(data="/content/dataset/data.yaml", split="val", plots=False)
print(f"\nAll validation images  mAP50 = {m.box.map50:.3f}")

own = sorted(glob.glob("/content/dataset/valid/images/own_webcam_photos_*"))
with open("/content/own_valid.txt", "w") as f:
    f.write("\n".join(own))
with open("/content/own_valid.yaml", "w") as f:
    f.write("path: /content/dataset\ntrain: train/images\nval: /content/own_valid.txt\n"
            "nc: 5\nnames: ['can', 'glasses', 'marker', 'watch', 'water bottle']\n")
m2 = best.val(data="/content/own_valid.yaml", split="val", plots=False)
print(f"\nYour webcam photos ({len(own)} images)  mAP50 = {m2.box.map50:.3f}")
recall = dict(zip(m2.box.ap_class_index, m2.box.r))
for i, name in best.names.items():
    print(f"  {name:13} recall {recall[i]:.2f}" if i in recall else f"  {name:13} (no photos)")

## 5. Export to ONNX and download

In [ ]:
onnx_path = best.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
final = "/content/5_objects_detector.onnx"
shutil.copy(onnx_path, final)
from google.colab import files
files.download(final)